In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Data Handling

## Dataset Construction

In [ ]:
from src.datasets.multi_season_dataset import build_multi_season_dataset


raw_X, raw_y, final_team_states_df = build_multi_season_dataset(
    seasons=list(range(2003, 2027))
)

print(f"\nX shape: {raw_X.shape}")
print(f"Number of features: {len(raw_X.columns)}")
print("Train features:")
print(list(raw_X.columns))

In [ ]:
from pathlib import Path

data_dir = Path("../../data")

final_team_states_df.to_csv(
    data_dir / "final_team_states.csv",
    index=False,
)

print("Saved to:", data_dir / "final_team_states.csv")

## Split

In [ ]:
from src.preprocessing.splits import split_raw_by_season

# Hold out the 2026 season as the test set
raw_X_train, raw_X_test, raw_y_train, raw_y_test = split_raw_by_season(
    raw_X,
    raw_y,
    test_seasons=[2026],
)

print("Train shape:", raw_X_train.shape)
print("Test shape:", raw_X_test.shape)

## Feature Engineering

In [ ]:
# from src.feature_engineering.feature_analysis import FeatureAnalyzer

# # Separate analyzers:
# # training data is used to make feature-engineering decisions,
# # while test data only receives the same predetermined transformations.
# train_analyzer = FeatureAnalyzer(raw_X_train)
# test_analyzer = FeatureAnalyzer(raw_X_test)


# # ---------------------------------------------------------
# # 1. Correlation analysis — TRAIN ONLY
# # ---------------------------------------------------------

# corr = train_analyzer.analyze_collinearity(threshold=0.90)

# # Features chosen for removal based on the training-data analysis
# redundant_features = [
#     "team_1_last_10_avg_opponent_win_pct",
#     "team_2_last_10_avg_opponent_win_pct",
#     "team_1_last_5_avg_opponent_win_pct",
#     "team_2_last_5_avg_opponent_win_pct",
#     "team_1_wins",
#     "team_2_wins",
#     "team_1_losses",
#     "team_2_losses",
# ]

# # Apply the same removals to train and test
# for analyzer in [train_analyzer, test_analyzer]:
#     analyzer.remove(redundant_features)


# # ---------------------------------------------------------
# # 2. Interaction analysis — TRAIN ONLY
# # ---------------------------------------------------------

# interactions = train_analyzer.analyze_interactions(
#     raw_y_train,
#     top_features=50
# )


# # ---------------------------------------------------------
# # 3. Create chosen interaction features identically
# #    in train and test
# # ---------------------------------------------------------

# for analyzer in [train_analyzer, test_analyzer]:

#     # Win percentage weighted by amount of season information available
#     analyzer.interaction(
#         "team_1_games_played",
#         "team_1_win_pct",
#         "team_1_wins_ratio",
#         "exponent"
#     )

#     analyzer.interaction(
#         "team_2_games_played",
#         "team_2_win_pct",
#         "team_2_wins_ratio",
#         "exponent"
#     )

#     # Historical location performance relevant to current matchup
#     analyzer.interaction(
#         "team_1_home_win_pct",
#         "team_1_location",
#         "team_1_home_strength",
#         "location"
#     )

#     analyzer.interaction(
#         "team_1_neutral_win_pct",
#         "team_1_location",
#         "team_1_neutral_strength",
#         "location"
#     )

#     analyzer.interaction(
#         "team_1_away_win_pct",
#         "team_1_location",
#         "team_1_away_strength",
#         "location"
#     )

#     analyzer.interaction(
#         "team_2_home_win_pct",
#         "team_1_location",
#         "team_2_home_strength",
#         "location"
#     )

#     analyzer.interaction(
#         "team_2_neutral_win_pct",
#         "team_1_location",
#         "team_2_neutral_strength",
#         "location"
#     )

#     analyzer.interaction(
#         "team_2_away_win_pct",
#         "team_1_location",
#         "team_2_away_strength",
#         "location"
#     )


# # ---------------------------------------------------------
# # 4. Remove base features replaced by engineered features
# # ---------------------------------------------------------

# replaced_features = [
#     "team_1_games_played",
#     "team_1_win_pct",
#     "team_2_games_played",
#     "team_2_win_pct",

#     "team_1_home_win_pct",
#     "team_1_neutral_win_pct",
#     "team_1_away_win_pct",

#     "team_2_home_win_pct",
#     "team_2_neutral_win_pct",
#     "team_2_away_win_pct",

#     "team_1_location",
# ]

# for analyzer in [train_analyzer, test_analyzer]:
#     analyzer.remove(replaced_features)


# # ---------------------------------------------------------
# # 5. Retrieve engineered train/test datasets
# # ---------------------------------------------------------

# raw_X_train = train_analyzer.get_data()
# raw_X_test = test_analyzer.get_data()

# # Sanity check: train and test must now have identical schemas
# assert raw_X_train.columns.tolist() == raw_X_test.columns.tolist()

# print("Engineered train shape:", raw_X_train.shape)
# print("Engineered test shape:", raw_X_test.shape)

In [ ]:
from src.feature_engineering.feature_analysis import FeatureAnalyzer

# Training data determines feature-engineering decisions.
# Test data only receives the same fixed transformations.
train_analyzer = FeatureAnalyzer(raw_X_train)
test_analyzer = FeatureAnalyzer(raw_X_test)


# ---------------------------------------------------------
# 1. Correlation analysis — TRAIN ONLY
# ---------------------------------------------------------

corr = train_analyzer.analyze_collinearity(threshold=0.90)

redundant_features = [
    "team_1_last_10_avg_opponent_win_pct",
    "team_2_last_10_avg_opponent_win_pct",
    "team_1_last_5_avg_opponent_win_pct",
    "team_2_last_5_avg_opponent_win_pct",
    "team_1_wins",
    "team_2_wins",
    "team_1_losses",
    "team_2_losses",
]

for analyzer in [train_analyzer, test_analyzer]:
    analyzer.remove(redundant_features)


# ---------------------------------------------------------
# 2. Create chosen interaction features
#    identically in train and test
# ---------------------------------------------------------

for analyzer in [train_analyzer, test_analyzer]:

    analyzer.interaction(
        "team_1_games_played",
        "team_1_win_pct",
        "team_1_wins_ratio",
        "exponent"
    )

    analyzer.interaction(
        "team_2_games_played",
        "team_2_win_pct",
        "team_2_wins_ratio",
        "exponent"
    )

    analyzer.interaction(
        "team_1_home_win_pct",
        "team_1_location",
        "team_1_home_strength",
        "location"
    )

    analyzer.interaction(
        "team_1_neutral_win_pct",
        "team_1_location",
        "team_1_neutral_strength",
        "location"
    )

    analyzer.interaction(
        "team_1_away_win_pct",
        "team_1_location",
        "team_1_away_strength",
        "location"
    )

    analyzer.interaction(
        "team_2_home_win_pct",
        "team_1_location",
        "team_2_home_strength",
        "location"
    )

    analyzer.interaction(
        "team_2_neutral_win_pct",
        "team_1_location",
        "team_2_neutral_strength",
        "location"
    )

    analyzer.interaction(
        "team_2_away_win_pct",
        "team_1_location",
        "team_2_away_strength",
        "location"
    )


# ---------------------------------------------------------
# 3. Remove base features replaced by interactions
# ---------------------------------------------------------

replaced_features = [
    "team_1_games_played",
    "team_1_win_pct",
    "team_2_games_played",
    "team_2_win_pct",

    "team_1_home_win_pct",
    "team_1_neutral_win_pct",
    "team_1_away_win_pct",

    "team_2_home_win_pct",
    "team_2_neutral_win_pct",
    "team_2_away_win_pct",

    "team_1_location",
]

for analyzer in [train_analyzer, test_analyzer]:
    analyzer.remove(replaced_features)


# ---------------------------------------------------------
# 4. Retrieve engineered datasets
# ---------------------------------------------------------

raw_X_train = train_analyzer.get_data()
raw_X_test = test_analyzer.get_data()

assert raw_X_train.columns.tolist() == raw_X_test.columns.tolist()

print("Engineered train shape:", raw_X_train.shape)
print("Engineered test shape:", raw_X_test.shape)

## Preprocessing

In [ ]:
from src.preprocessing.preprocess import preprocess_train_test

X_train, X_test, y_train, y_test = preprocess_train_test(
    raw_X_train,
    raw_X_test,
    raw_y_train,
    raw_y_test,
    prefix1="team_1_",
    prefix2="team_2_",
    invert_cols=["team_1_location"],
    diff_suffix="_diff",
    drop_base_features=True,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nNumber of model features:", X_train.shape[1])
print("Train/Test columns identical:",
      X_train.columns.tolist() == X_test.columns.tolist())

## Feature Selection

In [ ]:
from src.feature_engineering.feature_selector import FeatureSelector
import warnings

# ---------------------------------------------------------
# Final feature selection — TRAIN ONLY
# ---------------------------------------------------------

selector = FeatureSelector(X_train, y_train)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="sklearn.linear_model"
)
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="sklearn.linear_model"
)

# 1. Lasso
selector.run_lasso(
    target_features=35,
    print_results=True,
    plot_results=True
)

# 2. Recursive Feature Elimination
selector.run_rfe(
    target_features=35,
    print_results=True,
    plot_results=True
)

# 3. Tree-based feature importance
selector.run_tree_importance(
    target_features=35,
    print_results=True,
    plot_results=True
)

# 4. Keep features supported by at least two methods
selector.finalize_selection(
    vote_threshold=2,
    print_results=True
)

# 5. Apply the SAME selected feature set to train and test
X_train = selector.transform(X_train)
X_test = selector.transform(X_test)

# Sanity checks
assert X_train.columns.tolist() == X_test.columns.tolist()

print("\nFinal X_train shape:", X_train.shape)
print("Final X_test shape:", X_test.shape)
print("Final number of features:", X_train.shape[1])

In [ ]:
from pathlib import Path
import json

artifacts_dir = Path("../../artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

feature_columns = X_train.columns.tolist()

with open(artifacts_dir / "feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=4)

print(f"Saved {len(feature_columns)} selected features.")

## Save final datasets

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("../../data")

# Save final train/test feature matrices
X_train.to_csv(data_dir / "X_train.csv", index=False)
X_test.to_csv(data_dir / "X_test.csv", index=False)

# Save labels
y_train.to_csv(data_dir / "y_train.csv", index=False)
y_test.to_csv(data_dir / "y_test.csv", index=False)

# Save final selected feature names
pd.Series(X_train.columns, name="feature").to_csv(
    data_dir / "selected_features.csv",
    index=False
)

print("Saved:")
print(data_dir / "X_train.csv")
print(data_dir / "X_test.csv")
print(data_dir / "y_train.csv")
print(data_dir / "y_test.csv")
print(data_dir / "selected_features.csv")

print("\nShapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# Model Building

In [ ]:
from src.predictors.predictors import EnsembleLearner
from src.evaluation.evaluate_classifier import evaluate_classifier, print_classifier_results


def print_evaluation(sub_model, X_train, X_test, y_train, y_test):
    sub_model_eval = clone(sub_model)
    sub_model_eval.fit(X_train, y_train)
    results = evaluate_classifier(
        sub_model_eval,
        X_train,
        X_test,
        y_train,
        y_test,
    )
    print_classifier_results(results)

model = EnsembleLearner(meta_test_size=0.15)
print("Initializing model...")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_regression = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        C=0.1,
        solver="lbfgs",
        max_iter=10000
    ))
])

print_evaluation(
    logistic_regression,
    X_train,
    X_test,
    y_train,
    y_test
)

model.add_learner(
    "Logistic Regression (L2)",
    logistic_regression
)

In [ ]:
from lightgbm import LGBMClassifier

# Add LightGBM (The dominant algorithm for Men's data)
lightgbm = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.02,
    max_depth=5,
    num_leaves=31,
    min_child_samples=40,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    random_state=42,
    verbose=-1,
)
print_evaluation(lightgbm, X_train, X_test, y_train, y_test)
model.add_learner("LightGBM", lightgbm)